In [ ]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

In [ ]:
%autoreload
import torch
import torch.nn as nn
from torch.optim import SGD, Adam
from source.process.data import trainLoader, scoreLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from source.model.model import ResNext100
from source.model.train import trainModel
from source.utils.loss import FocalLoss, F2Loss

In [ ]:
def step_1(fold):
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    loader = {}
    loader['image_path'] = '../../data/train/'
    loader['label_path'] = '../../data/folds.csv'
    loader['batch'] = 20
    loader['size'] = 300
    loader['fold_idx'] = fold
    train_data, valid_data = trainLoader(**loader)
    model = ResNext100(freeze=True).to(device)
    optimizer = Adam(model.parameters(), lr=1e-3)
    schedular = CosineAnnealingLR(optimizer, 1, eta_min=1e-3)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train_data
    trainer['valid_data'] = valid_data
    trainer['loss_fn'] = FocalLoss(gamma=1.)
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../resnext100/stage_1_{}.pt'.format(fold)
    trainer['load_path'] = None
    trainer['epochs'] = 1
    trainer['batch'] = 20
    trainer['scheduler'] = schedular
    trainer['device'] = device
    trainModel(**trainer)
    model.cpu()
    del model
    return None

def step_2(fold):
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    loader = {}
    loader['image_path'] = '../../data/train/'
    loader['label_path'] = '../../data/folds.csv'
    loader['batch'] = 20
    loader['size'] = 300
    loader['fold_idx'] = fold
    train_data, valid_data = trainLoader(**loader)
    model = ResNext100(freeze=False).to(device)
    optimizer = Adam(model.parameters(), lr=1e-4)
    schedular = CosineAnnealingLR(optimizer, 1, eta_min=1e-5)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train_data
    trainer['valid_data'] = valid_data
    trainer['loss_fn'] = FocalLoss(gamma=1.)
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../resnext100/stage_2_{}.pt'.format(fold)
    trainer['load_path'] = '../../resnext100/stage_1_{}.pt'.format(fold)
    trainer['epochs'] = 20
    trainer['batch'] = 20
    trainer['scheduler'] = schedular
    trainer['device'] = device
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
for idx in [1,2,3,4]:
    step_1(idx)
    step_2(idx)